In [10]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack


In [11]:
# ==========================================
# Task 1: Data Preprocessing
# ==========================================

# 1. Load the dataset
anime_df = pd.read_csv('/content/anime.csv')

print(anime_df.head())
print(anime_df.info())
print(anime_df.describe())


   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  

In [12]:
anime_df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [13]:
# 2. Handle missing values
# Drop rows where 'genre' or 'name' is missing, as they are crucial for content-based similarity
anime_df = anime_df.dropna(subset=['genre', 'name'])

# Fill missing ratings with the median rating
anime_df['rating'] = anime_df['rating'].fillna(anime_df['rating'].median())

# Fill missing broadcast types with 'Unknown'
anime_df['type'] = anime_df['type'].fillna('Unknown')

# Reset the index after dropping rows to ensure clean lookups
anime_df = anime_df.reset_index(drop=True)

In [14]:
# ==========================================
# Task 2: Feature Extraction
# ==========================================

# 1. Genres (Categorical to Numerical using TF-IDF)
# Using TF-IDF helps assign higher weights to unique/rare genres compared to common ones like "Comedy"
tfidf = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b') # Tokenize by word
genre_matrix = tfidf.fit_transform(anime_df['genre'].str.replace(', ', ' '))

# 2. Rating & Members (Numerical Normalization)
# Normalize so that large member counts don't dominate the similarity score
scaler = MinMaxScaler()
num_features = scaler.fit_transform(anime_df[['rating', 'members']])

# Combine the textual genre features with the normalized numerical features
# Convert to a Compressed Sparse Row (CSR) matrix for efficient row-wise operations
combined_features = hstack([genre_matrix, num_features]).tocsr()


In [15]:
# ==========================================
# Task 3: Recommendation System
# ==========================================

def get_recommendations(title, df, feature_matrix, threshold=None, top_n=5):
    """
    Given a target anime title, recommends similar anime based on cosine similarity.
    Allows for a minimum similarity threshold to adjust the list size/quality.
    """
    # 1. Find the index of the target anime
    try:
        idx = df[df['name'].str.lower() == title.lower()].index[0]
    except IndexError:
        return "Anime not found in dataset."

    # 2. Compute cosine similarity between the target anime and ALL other anime
    # Flatten the result to a 1D array
    sim_scores = cosine_similarity(feature_matrix[idx], feature_matrix).flatten()

    # 3. Get the indices of the anime sorted by similarity score (descending order)
    anime_indices = np.argsort(sim_scores)[::-1]

    results = []
    # 4. Iterate over sorted indices, skipping the first one (which is the target anime itself)
    for i in anime_indices[1:]:
        score = sim_scores[i]

        # Apply the similarity threshold if provided
        if threshold is not None and score < threshold:
            continue

        results.append({
            'Name': df.iloc[i]['name'],
            'Genre': df.iloc[i]['genre'],
            'Similarity Score': round(score, 4)
        })

        # Stop once we hit our desired top_n count
        if len(results) >= top_n:
            break

    return pd.DataFrame(results)

print("Recommendations for 'Naruto' (No threshold, Top 5):")
print(get_recommendations('Naruto', anime_df, combined_features, top_n=5))

print("\n" + "="*60 + "\n")

print("Recommendations for 'Death Note' (Strict Threshold of 0.85):")
# Here, if we set the threshold to 0.85, the size of the recommendation list adjusts dynamically
# if fewer than top_n anime meet the criteria.
print(get_recommendations('Death Note', anime_df, combined_features, threshold=0.85, top_n=5))

Recommendations for 'Naruto' (No threshold, Top 5):
                                                Name  \
0                                 Naruto: Shippuuden   
1                                      Dragon Ball Z   
2                                        Dragon Ball   
3        Naruto: Shippuuden Movie 4 - The Lost Tower   
4  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   

                                               Genre  Similarity Score  
0  Action, Comedy, Martial Arts, Shounen, Super P...            0.9951  
1  Action, Adventure, Comedy, Fantasy, Martial Ar...            0.9428  
2  Adventure, Comedy, Fantasy, Martial Arts, Shou...            0.9157  
3  Action, Comedy, Martial Arts, Shounen, Super P...            0.9092  
4  Action, Comedy, Martial Arts, Shounen, Super P...            0.9088  


Recommendations for 'Death Note' (Strict Threshold of 0.85):
               Name                                              Genre  \
0  Mirai Nikki (TV)  Action, Mystery

**1) What is Collaborative Filtering?**

Collaborative Filtering is a recommendation technique used to predict a user’s preferences based on the behavior of similar users.

It works on the assumption that:

“Users who agreed in the past will likely agree in the future.”

It uses user–item interaction data such as ratings, clicks, purchases, or watch history.

There are two major types:

* User-Based Collaborative Filtering
* Item-Based Collaborative Filtering

---

**How Collaborative Filtering Works (Step-by-Step):**

1. Create a User-Item Matrix
   Rows represent users, columns represent items, and values represent ratings or interactions.

2. Compute Similarity
   Similarity between users or items is calculated using:

   * Cosine Similarity
   * Pearson Correlation
   * Euclidean Distance

3. Predict Ratings
   Based on similar users or similar items, the system predicts missing ratings.

4. Recommend Items
   Items with the highest predicted ratings are recommended.

Example:
If User A and User B have similar movie ratings, and User B liked a movie that User A hasn’t seen, that movie will be recommended to User A.

---

**2) Difference Between User-Based and Item-Based Collaborative Filtering**

User-Based Collaborative Filtering:

* Focuses on similarity between users.
* Finds users similar to the target user.
* Recommends items liked by similar users.
* Works well when the number of users is small.
* Can be computationally expensive for large datasets.

Example Logic:
“Users similar to you liked this item.”

---

Item-Based Collaborative Filtering:

* Focuses on similarity between items.
* Finds items similar to what the user already liked.
* Recommends similar items.
* More scalable and efficient for large systems.
* Used widely in real-world platforms like Netflix and Amazon.

Example Logic:
“Items similar to what you liked are recommended.”

---

**Key Differences Summary**

User-Based:

* Similarity between users
* Dynamic user behavior
* Less scalable with many users

Item-Based:

* Similarity between items
* Items are relatively stable
* More scalable and efficient

---

**Additional Important Points for Interviews**

* Cold Start Problem: New users or new items have no data, so recommendations are difficult.
* Data Sparsity: Most users rate only a few items.
* Matrix Factorization: Advanced technique used in large-scale systems.
* Collaborative Filtering does not use item content (unlike content-based filtering).


